<a href="https://colab.research.google.com/github/OkktaDan/CCMACLRL_EXERCISES_COM231/blob/main/Exercises/Exercise_4/Octaviano_Exercise4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 4: Hate Speech Classification using Multinomial Naive Bayes

Instructions:
- You do not need to split your data. Use the training, validation and test sets provided below.
- Use Multinomial Naive Bayes to train a model that can classify if a sentence is a hate speech or non-hate speech
- A sentence with a label of zero (0) is classified as non-hate speech
- A sentence with a label of one (1) is classified as a hate speech

Apply text pre-processing techniques such as
- Converting to lowercase
- Stop word Removal
- Removal of digits, special characters
- Stemming or Lemmatization but not both
- Count Vectorizer or TF-IDF Vectorizer but not both

Evaluate your model by:
- Providing input by yourself
- Creating a Confusion Matrix
- Calculating the Accuracy, Precision, Recall and F1-Score

In [156]:
import pandas as pd
import numpy as np
import nltk
import re
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords



nltk.download("stopwords")
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [157]:
splits = {'train': 'unique_train_dataset.csv', 'validation': 'unique_validation_dataset.csv', 'test': 'unique_test_dataset.csv'}

**Training Set**

Use this to train your model

In [158]:
df_train = pd.read_csv("hf://datasets/mapsoriano/2016_2022_hate_speech_filipino/" + splits["train"])

**Validation Set**

Use this set to evaluate your model

In [159]:
df_validation = pd.read_csv("hf://datasets/mapsoriano/2016_2022_hate_speech_filipino/" + splits["validation"])

**Test Set**
  
Use this set to test your model

In [160]:
df_test = pd.read_csv("hf://datasets/mapsoriano/2016_2022_hate_speech_filipino/" + splits["test"])

## A. Understanding your training data

1. Check the first 10 rows of the training dataset

In [161]:
df_train.head(10)

,text,label
0,Presidential candidate Mar Roxas implies that ...,1
1,Parang may mali na sumunod ang patalastas ng N...,1
2,Bet ko. Pula Ang Kulay Ng Posas,1
3,[USERNAME] kakampink,0
4,Bakit parang tahimik ang mga PINK about Doc Wi...,1
5,"""Ang sinungaling sa umpisa ay sinungaling hang...",1
6,Leni Kiko,0
7,Nahiya si Binay sa Makati kaya dito na lang sa...,1
8,Another reminderHalalan,0
9,[USERNAME] Maybe because VP Leni Sen Kiko and ...,0


2. Check how many rows and columns are in the training dataset using `.info()`

In [162]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21773 entries, 0 to 21772
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    21773 non-null  object
 1   label   21773 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 340.3+ KB


3. Check for NaN values

In [163]:
df_train.isnull().sum()

,0
text,0
label,0


4. Check for duplicate rows

In [164]:
df_train.duplicated().sum()

np.int64(0)

5. Check how many rows belong to each class

In [165]:
df_train.groupby('label').describe()

text                                                               
       count unique                                                top freq
label                                                                      
0      10779  10779  [USERNAME] Cnu ba naman ang hindImabibighani s...    1
1      10994  10994  Honestly I dont get kakampinks like pano nila ...    1

In [166]:
df_train['length'] = df_train['text'].apply(len)
df_train = df_train[['text', 'length', 'label']]
df_train.head()

,text,length,label
0,Presidential candidate Mar Roxas implies that ...,116,1
1,Parang may mali na sumunod ang patalastas ng N...,107,1
2,Bet ko. Pula Ang Kulay Ng Posas,31,1
3,[USERNAME] kakampink,20,0
4,Bakit parang tahimik ang mga PINK about Doc Wi...,74,1


## B. Text pre-processing

6. Remove duplicate rows

In [167]:
if (df_train.duplicated().any().any()):
  df_train.drop_duplicates(inplace=True)
else:
  print("No detected duplicates")


No detected duplicates


7. Remove rows with NaN values

In [168]:
if (df_train.isnull().any().any()):
  df_train.dropna(inplace=True)
else:
  print("No detected rows with NaN values")


No detected rows with NaN values


8. Convert all text to lowercase

In [169]:
df_train['text'] = df_train['text'].str.lower()
df_train.head(10)

,text,length,label
0,presidential candidate mar roxas implies that ...,116,1
1,parang may mali na sumunod ang patalastas ng n...,107,1
2,bet ko. pula ang kulay ng posas,31,1
3,[username] kakampink,20,0
4,bakit parang tahimik ang mga pink about doc wi...,74,1
5,"""ang sinungaling sa umpisa ay sinungaling hang...",122,1
6,leni kiko,9,0
7,nahiya si binay sa makati kaya dito na lang sa...,72,1
8,another reminderhalalan,23,0
9,[username] maybe because vp leni sen kiko and ...,253,0


9. Remove digits, URLS and special characters

In [170]:
df_train['text'] = (df_train['text'].str.replace(r'http\S+|www\S+|https\S+', '', regex=True).str.replace(r'\d+', '', regex=True).str.replace(r'[^A-Za-z\s]', '', regex=True).str.strip())
df_train.head(10)

,text,length,label
0,presidential candidate mar roxas implies that ...,116,1
1,parang may mali na sumunod ang patalastas ng n...,107,1
2,bet ko pula ang kulay ng posas,31,1
3,username kakampink,20,0
4,bakit parang tahimik ang mga pink about doc wi...,74,1
5,ang sinungaling sa umpisa ay sinungaling hangg...,122,1
6,leni kiko,9,0
7,nahiya si binay sa makati kaya dito na lang sa...,72,1
8,another reminderhalalan,23,0
9,username maybe because vp leni sen kiko and th...,253,0


10. Remove stop words

In [171]:

fileng_stopwords = {
    "ang", "ng", "sa", "ay", "na", "mga", "para", "ito", "iyon", "ating", "lahat",
    "at", "si", "ni", "kay", "kanyang", "kaniya", "nila", "natin", "mo", "ko", "ka",
    "kami", "ikaw", "ako", "sila", "niya", "ngayon", "doon", "dito", "roon", "rito",
    "ngunit", "dahil", "kapag", "kung", "habang", "upang", "mula", "hanggang", "pa",
    "lamang", "rin", "din", "man", "pati", "sa", "ng", "na", "o", "pero", "kaya", "iyon",
    "ating", "ilang", "isa", "ilang", "lahat", "walang", "may", "meron", "bawat",
    "mas", "masyado", "maging", "nang", "naman", "nga", "ng", "ni", "o", "pa", "pala",
    "saan", "sabi", "sabihin", "tulad", "tungkol", "wala", "walaing", "walaon", "yaon",
    "ayon", "ata", "a", "about", "above", "after", "again", "against", "all", "am", "an", "and",
    "any", "are", "aren't", "as", "at", "be", "because", "been", "before", "being",
    "below", "between", "both", "but", "by", "can't", "cannot", "could", "couldn't",
    "did", "didn't", "do", "does", "doesn't", "doing", "don't", "down", "during", "each",
    "few", "for", "from", "further", "had", "hadn't", "has", "hasn't", "have", "haven't",
    "having", "he", "he'd", "he'll", "he's", "her", "here", "here's", "hers", "herself",
    "him", "himself", "his", "how", "how's", "i", "i'd", "i'll", "i'm", "i've", "if",
    "in", "into", "is", "isn't", "it", "it's", "its", "itself", "let's", "me", "more",
    "most", "mustn't", "my", "myself", "no", "nor", "not", "of", "off", "on", "once",
    "only", "or", "other", "ought", "our", "ours", "ourselves", "out", "over", "own",
    "same", "shan't", "she", "she'd", "she'll", "she's", "should", "shouldn't", "so",
    "some", "such", "than", "that", "that's", "the", "their", "theirs", "them", "themselves",
    "then", "there", "there's", "these", "they", "they'd", "they'll", "they're", "they've",
    "this", "those", "through", "to", "too", "under", "until", "up", "very", "was", "wasn't",
    "we", "we'd", "we'll", "we're", "we've", "were", "weren't", "what", "what's", "when",
    "when's", "where", "where's", "which", "while", "who", "who's", "whom", "why", "why's",
    "with", "won't", "would", "wouldn't", "you", "you'd", "you'll", "you're", "you've",
    "your", "yours", "yourself", "yourselves", "maybe", "get", "every"
}

def remve_stopwords(text):
    words = text.split()
    filtered = [word for word in words if word not in list(fileng_stopwords)]
    return " ".join(filtered)

#sample
messageo = df_train.text[9]
print(f"original: {messageo}")
message = remve_stopwords(messageo)
print(f"removed stopwords: {message}")



original: username maybe because vp leni sen kiko and the whole team are buoyed and energized by the very strong support they get in every sorties aside from their strong commitment on the fight laban para sa ating bayan angat buhay lahat sa gobyernong tapat
removed stopwords: username vp leni sen kiko whole team buoyed energized strong support sorties aside strong commitment fight laban bayan angat buhay gobyernong tapat


In [172]:
df_train['text'] = df_train['text'].apply(remve_stopwords)
df_train.head(100)

,text,length,label
0,presidential candidate mar roxas implies govt ...,116,1
1,parang mali sumunod patalastas nescaf coffee b...,107,1
2,bet pula kulay posas,31,1
3,username kakampink,20,0
4,bakit parang tahimik pink doc willie ong react...,74,1
...,...,...,...
95,popogi anak ngmga kandidato yung binay lang hindi,64,1
96,urrrggggbinay aldub happy bday lola,36,1
97,presidential aspirants vice president username...,219,0
98,leni president usernameusername never marcos m...,66,1


11. Use Stemming or Lemmatization

In [173]:
lemmatizer = WordNetLemmatizer()
def lemmatize_word(text):
  return " ".join([lemmatizer.lemmatize(word, pos='v') for word in text.split()])

messageo = df_train.text[9]
print(f"original: {messageo}")
message = lemmatize_word(messageo)
print(f"lemmatize: {message}")

original: username vp leni sen kiko whole team buoyed energized strong support sorties aside strong commitment fight laban bayan angat buhay gobyernong tapat
lemmatize: username vp leni sen kiko whole team buoy energize strong support sorties aside strong commitment fight laban bayan angat buhay gobyernong tapat


In [174]:
df_train['text'] = df_train['text'].apply(lemmatize_word)
df_train.head(100)

,text,length,label
0,presidential candidate mar roxas imply govt li...,116,1
1,parang mali sumunod patalastas nescaf coffee b...,107,1
2,bet pula kulay posas,31,1
3,username kakampink,20,0
4,bakit parang tahimik pink doc willie ong react...,74,1
...,...,...,...
95,popogi anak ngmga kandidato yung binay lang hindi,64,1
96,urrrggggbinay aldub happy bday lola,36,1
97,presidential aspirants vice president username...,219,0
98,leni president usernameusername never marcos m...,66,1


## C. Training your model

12. Put all text training data in variable **X_train**

In [175]:
x_train = df_train['text']
x_train

,text
0,presidential candidate mar roxas imply govt li...
1,parang mali sumunod patalastas nescaf coffee b...
2,bet pula kulay posas
3,username kakampink
4,bakit parang tahimik pink doc willie ong react...
...,...
21768,marcos talunan marcos magnanakaw
21769,grabe kayo binay
21770,username cnu ba hindimabibighani iyo maamkakam...
21771,rt username tabi tabi yung nagsasabing parang ...


13. Put all training data labels in variable **y_train**

In [176]:
y_train = df_train['label']
y_train

,label
0,1
1,1
2,1
3,0
4,1
...,...
21768,1
21769,0
21770,0
21771,1


14. Use `CountVectorizer()` or `TfidfVectorizer()` to convert text data to its numerical form.

Put the converted data to **X_train_transformed** variable

In [177]:
vect = TfidfVectorizer(stop_words=list(fileng_stopwords), max_df=0.5)
x_train_transformed = vect.fit_transform(x_train)


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aren', 'can', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'let', 'll', 'mustn', 're', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


15. Create an instance of `MultinomalNB()`

In [178]:
nb = MultinomialNB()

16. Train the model using `.fit()`

In [179]:
nb.fit(x_train_transformed,y_train)

MultinomialNB()

## D. Evaluate your model

17. Use `.predict()` to generate model predictions using the **validation dataset**


- Put all text validation data in **X_validation** variable

- Convert **X_validation** to its numerical form.

- Put the converted data to **X_validation_transformed**

- Put all predictions in **y_validation_pred** variable

In [180]:
df_validation['text'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 2800 entries, 0 to 2799
Series name: text
Non-Null Count  Dtype 
--------------  ----- 
2800 non-null   object
dtypes: object(1)
memory usage: 22.0+ KB


In [181]:
x_validation = df_validation['text']
x_validation_transformed = vect.transform(x_validation)

y_validation_pred = nb.predict(x_validation_transformed)


18. Get the Accuracy, Precision, Recall and F1-Score of the model using the **validation dataset**

- Put all validation data labels in **y_validation** variable

In [182]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
y_validation = df_validation['label']

print(f"ACCOURACY: {accuracy_score(y_validation, y_validation_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_validation, y_validation_pred, average='macro')*100:.2f}%")
print(f"Recall: {recall_score(y_validation, y_validation_pred, average='macro')*100:.2f}%")
print(f"F1-score: {f1_score(y_validation, y_validation_pred, average='macro')*100:.2f}%")

"""
Key intake:
  - So i tried making my own list of stop words since the stopword list provided only had 'english' stopwords.
  I took the chance to experiment and mixed both english and tagalog words. The current printed results are a mix of english and tagalog stopwords.


METRICS IF I USED ENGLISH ONLY STOP WORDS:
# ACCOURACY: 83.50%
# Precision: 83.91%
# Recall: 83.44%
# F1-score: 83.43%
"""


ACCOURACY: 83.36%
Precision: 83.55%
Recall: 83.32%
F1-score: 83.32%


"\nKey intake:\n  - So i tried making my own list of stop words since the stopword list provided only had 'english' stopwords.\n  I took the chance to experiment and mixed both english and tagalog words. The current printed results are a mix of english and tagalog stopwords.\n\n\nMETRICS IF I USED ENGLISH ONLY STOP WORDS:\n# ACCOURACY: 83.50%\n# Precision: 83.91%\n# Recall: 83.44%\n# F1-score: 83.43%\n"

19. Create a confusion matrix using the **validation dataset**

In [183]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_validation, y_validation_pred)
print(cm)

[[1101  284]
 [ 182 1233]]


20. Use `.predict()` to generate the model predictions using the **test dataset**


- Put all text validation data in **X_test** variable

- Convert **X_test** to its numerical form.

- Put the converted data to **X_test_transformed**

- Put all predictions in **y_test_pred** variable

In [184]:
x_test = df_test['text']
x_test_transformed = vect.transform(x_test)
y_test_pred = nb.predict(x_test_transformed)

21. Get the Accuracy, Precision, Recall and F1-Score of the model using the **test dataset**

- Put all test data labels in **y_validation** variable



In [185]:
y_test = df_test['label']

print(f"ACCOURACY: {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"Precision: {precision_score(y_test, y_test_pred, average='macro')*100:.2f}%")
print(f"Recall: {recall_score(y_test, y_test_pred, average='macro')*100:.2f}%")
print(f"F1-score: {f1_score(y_test, y_test_pred, average='macro')*100:.2f}%")

"""
Key intake:
  - So i tried making my own list of stop words since the stopword list provided only had 'english' stopwords.
  I took the chance to experiment and mixed both english and tagalog words. The current printed results are a mix of english and tagalog stopwords.


METRICS IF I USED ENGLISH ONLY STOP WORDS:
# ACCOURACY: 83.06
# Precision: 83.54%
# Recall: 83.09%
# F1-score: 83.01%
"""

ACCOURACY: 82.95%
Precision: 83.15%
Recall: 82.97%
F1-score: 82.93%


"\nKey intake:\n  - So i tried making my own list of stop words since the stopword list provided only had 'english' stopwords.\n  I took the chance to experiment and mixed both english and tagalog words. The current printed results are a mix of english and tagalog stopwords.\n\n\nMETRICS IF I USED ENGLISH ONLY STOP WORDS:\n# ACCOURACY: 83.06\n# Precision: 83.54%\n# Recall: 83.09%\n# F1-score: 83.01%\n"

22. Create a confusion matrix using the **test dataset**

In [186]:
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

[[1118  294]
 [ 185 1213]]


## E. Test the model

23. Test the model by providing a non-hate speech input. The model should predict it as 0

In [187]:
non_hate_speech = pd.Series('life is good')
non_hate_speech_transformed = vect.transform(non_hate_speech)
print("NONE-HATE:", nb.predict(non_hate_speech_transformed))

NONE-HATE: [0]


24. Test the model by providing a hate speech input. The model should predict it as 1

In [188]:
hate_speech = pd.Series('Marcos')
hate_speech_transformed = vect.transform(hate_speech)
print("HATE:", nb.predict(hate_speech_transformed))

HATE: [1]
